# OpenFlow — Kaggle GPU Worker

**Before running:**
1. Notebook settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
2. Attach your private Dataset with the 4 model files (unet .gguf, umt5 fp8 encoder, wan2.2 vae, lightning loras) — see README.
3. Set `BACKEND` below to your Cloudflare Tunnel URL (from the machine running the OpenFlow backend).

This worker polls the backend over HTTP, so no inbound ports are needed on Kaggle.

In [ ]:
BACKEND = "https://YOUR-TUNNEL.trycloudflare.com"  # <-- set me\nimport os; os.environ["BACKEND"] = BACKEND

In [ ]:
# 1. Pull the OpenFlow worker code\n!git clone https://github.com/HiepDuong03/997162.git /kaggle/working/openflow\n%cd /kaggle/working/openflow/worker

In [ ]:
# 2. Bootstrap ComfyUI + ComfyUI-GGUF + link models (5-15 min first run)
!bash kaggle_bootstrap.sh

In [ ]:
# 3. Start ComfyUI headless in the background
import subprocess, time, requests
COMFY = '/kaggle/working/ComfyUI'
proc = subprocess.Popen(['python', f'{COMFY}/main.py', '--listen', '127.0.0.1',
                         '--port', '8188', '--lowvram'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        requests.get('http://127.0.0.1:8188/system_stats', timeout=3); print('ComfyUI up'); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI did not start')

In [ ]:
# 4. Run the worker (polls backend, drives ComfyUI). One worker per GPU.
#    With 2x T4 you can run a second notebook cell/session pointing at a 2nd ComfyUI on port 8189.
!python worker.py --comfy --comfy-url http://127.0.0.1:8188